In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder, StandardScaler
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
#sc = StandardScaler()

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

#x_train = sc.fit_transform(X=x_train)
#x_dev = sc.transform(x_dev)


In [4]:
x_train.head()

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,35,200,0,6.82,1081.47,0.922,0.920
1,35,1500,0,52.25,2014.73,0.920,0.925
2,95,200,0,16.57,1321.93,0.912,0.924
3,95,200,0,41.42,1615.83,0.948,0.939
4,35,1500,0,63.82,1137.29,0.930,0.937


In [5]:
df = pd.read_csv("data/Data_RSW.csv")

In [6]:
print(x_dev.shape)

(99, 7)


In [7]:
print(len(x_train))

295


In [8]:
print(y_train)

0      2127.7
1      5346.4
2      2350.4
3      2174.8
4      3897.5
        ...  
290    3141.8
291    2312.3
292    3105.4
293    2730.3
294    2897.8
Name: PullTest (N), Length: 295, dtype: float64


In [9]:
print(x_dev)

    Pressure (PSI)  Welding Time (ms)  Angle (Deg)  Force (N)  Current (A)  \
0               95               1500            0     124.19      1045.90   
1               35                200            0       6.58      1171.55   
2               95               1500            0     124.16      1078.29   
3               35                200            0       6.61       916.16   
4               35               1500           15      54.11      1440.24   
..             ...                ...          ...        ...          ...   
94              60               1200            0      97.26      3624.99   
95              60               1200            0      97.29      4311.15   
96              60               1200            0      98.26      3490.11   
97              60               1200            0      98.15      3311.58   
98              60               1200            0      98.09      2944.84   

    Thickness A (mm)  Thickness B (mm)  
0              0.918  

In [10]:
x_train.columns

Index(['Pressure (PSI)', 'Welding Time (ms)', 'Angle (Deg)', 'Force (N)',
       'Current (A)', 'Thickness A (mm)', 'Thickness B (mm)'],
      dtype='object')

In [11]:
x_train.loc[x_train['Category']=='Bad']["NuggetDiameter (mm)"].min()

KeyError: 'Category'

In [ ]:
x_train.loc[x_train['Category']=='Bad']["NuggetDiameter (mm)"].max()

np.float64(2.7)

In [ ]:
print(x_train.loc[x_train['Category']=='Good']["NuggetDiameter (mm)"].min())
print(x_train.loc[x_train['Category']=='Good']["NuggetDiameter (mm)"].max())

2.83
4.57


In [ ]:
print(x_train.loc[x_train['Category']=='Explode']["NuggetDiameter (mm)"].min())
print(x_train.loc[x_train['Category']=='Explode']["NuggetDiameter (mm)"].max())

2.59
4.05


In [ ]:
p = []
for f in x_train.loc[x_train['Category']=='Good'].index:
    p.append(y_train[f])

In [ ]:
print(min(p), max(p))

2426.5 5634.8


In [ ]:
if list(y_train) == list(x_train['Force (N)'].values):
    print('eq')

In [ ]:
list(x_train['Force (N)'].values)

[np.float64(115.54),
 np.float64(8.31),
 np.float64(97.09),
 np.float64(93.01),
 np.float64(96.29),
 np.float64(91.87),
 np.float64(108.21),
 np.float64(114.26),
 np.float64(77.33),
 np.float64(115.91),
 np.float64(95.1),
 np.float64(73.43),
 np.float64(93.02),
 np.float64(93.85),
 np.float64(82.81),
 np.float64(77.54),
 np.float64(114.15),
 np.float64(91.9),
 np.float64(107.45),
 np.float64(79.88),
 np.float64(100.02),
 np.float64(80.22),
 np.float64(115.6),
 np.float64(96.05),
 np.float64(95.61),
 np.float64(73.95),
 np.float64(78.69),
 np.float64(92.53),
 np.float64(95.08),
 np.float64(108.2),
 np.float64(92.93),
 np.float64(115.91),
 np.float64(89.33),
 np.float64(52.25),
 np.float64(93.01),
 np.float64(93.07),
 np.float64(83.46),
 np.float64(91.89),
 np.float64(97.67),
 np.float64(114.95),
 np.float64(115.91),
 np.float64(93.19),
 np.float64(107.86),
 np.float64(97.17),
 np.float64(108.36),
 np.float64(80.29),
 np.float64(94.24),
 np.float64(95.81),
 np.float64(92.71),
 np.float64

# Fit Model

In [12]:
# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor.fit(x_train, y_train)

# Predict on the test set
predictions = regressor.predict(x_dev)

# Check Validation Data

In [13]:
# Convert to numpy arrays (if not already)
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(predictions).ravel()

# Sample index for plotting
sample_idx = np.arange(len(true_vals))

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sample_idx, y=true_vals, mode="markers",
    name="Original Values", marker=dict(color="red", size=6)
))

fig.add_trace(go.Scatter(
    x=sample_idx, y=pred_vals, mode="markers",
    name="Predicted Values", marker=dict(color="blue", size=6)
))

# Connecting lines (one per sample) 
for i in range(len(sample_idx)): 
    fig.add_trace(go.Scatter( 
        x=[sample_idx[i], sample_idx[i]], 
        y=[true_vals[i], pred_vals[i]], 
        mode="lines", 
        line=dict(color="gray", width=1), 
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN)",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.write_html("Graphs/TabPFN_Evaltrue.html")
fig.show()


In [ ]:
print(len(predictions))

NameError: name 'predictions' is not defined

# Check Validation Loss and R2

In [14]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(mean_squared_error(y_dev, predictions))
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  126.26
RMSE: 219.93
R2: 0.62
